# 06 — Evaluation and Ablation

This notebook performs the final experimental evaluation required by the project proposal.

Metrics:
- Precision
- Recall
- F1-score
- AUROC
- AUPRC
- Mean lead time

Ablation:
- Frequent patterns
- Closed patterns
- Discriminative patterns
- Optional classifier without sequential pattern features

The same evaluation protocol should be used across experiments.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
EVAL_DIR = PROJECT_ROOT / "outputs" / "evaluation"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

results_path = EVAL_DIR / "classification_results.csv"

if results_path.exists():
    results = pd.read_csv(results_path)
    display(results)
else:
    print("Run Notebook 05 first.")


## Lead time

Lead time measures how early a correct positive prediction occurs relative to the observed sepsis onset.

For a complete implementation, this should be calculated from timestamp/hour information rather than inferred from aggregate classification metrics.

The result should be reported in hours.

Do not fabricate lead-time values if the necessary prediction timeline has not been generated.


In [ ]:
# Lead-time calculation should consume saved patient-level prediction timelines.
# This placeholder intentionally fails clearly instead of inventing a result.

def calculate_mean_lead_time(prediction_timelines):
    if prediction_timelines is None:
        raise ValueError(
            "Prediction timelines are required to calculate mean lead time. "
            "Do not infer lead time from patient-level classification metrics."
        )

    lead_times = []
    for timeline in prediction_timelines:
        if timeline.get("correct_prediction") and timeline.get("sepsis_onset_hour") is not None:
            lead_times.append(
                timeline["sepsis_onset_hour"] - timeline["prediction_hour"]
            )

    return float(np.mean(lead_times)) if lead_times else np.nan


## Ablation framework

The ablation study should compare representations generated from:

1. Frequent patterns only
2. Closed patterns only
3. Discriminative patterns
4. Optional non-sequential baseline

The models should use the same patient split and evaluation metrics.

The purpose is to determine whether closed-pattern reduction and discriminative selection contribute to the final representation.


In [ ]:
# Expected schema for ablation results.
# Populate this table after running the corresponding experiments.
ablation_columns = [
    "experiment",
    "precision",
    "recall",
    "f1",
    "auroc",
    "auprc",
    "mean_lead_time_hours",
]

ablation_results = pd.DataFrame(columns=ablation_columns)

ablation_results.to_csv(
    EVAL_DIR / "ablation_results.csv",
    index=False
)

print("Ablation result schema created.")


In [ ]:
# Example visualization code for completed ablation results.
ablation_path = EVAL_DIR / "ablation_results.csv"

if ablation_path.exists():
    ablation = pd.read_csv(ablation_path)

    if not ablation.empty:
        ax = ablation.set_index("experiment")[["recall", "f1", "auprc"]].plot(
            kind="bar",
            figsize=(10, 5)
        )
        ax.set_ylabel("Score")
        ax.set_title("Ablation Study")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "ablation_metrics.png", dpi=200)
        plt.show()
    else:
        print("Ablation table is empty; run the experiments first.")
